In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os

os.makedirs("../output", exist_ok=True)

panel = pd.read_csv("../data/stacked_event_panel.csv")
panel = panel[(panel["k"] >= -6) & (panel["k"] <= 12)].copy()
panel["post"] = (panel["k"] >= 0).astype(int)
panel["t_idx"] = panel.groupby("state_event")["month"].rank(method="dense")

specs = {
    "(1) Basic": "log_handle ~ post + C(state_event)",
    "(2) + state-event linear trend": "log_handle ~ post + C(state_event) + C(state_event):t_idx",
}

print("=" * 100)
print("STATIC PRE/POST SPECIFICATION (real data, N=%d)" % len(panel))
print("=" * 100)

results = {}
for label, formula in specs.items():
    m_cluster = smf.ols(formula, data=panel).fit(cov_type="cluster", cov_kwds={"groups": panel["state"]})
    m_hc1 = smf.ols(formula, data=panel).fit(cov_type="HC1")
    results[label] = (m_cluster, m_hc1)
    print(f"\n{label}")
    print(f"  post coefficient : {m_cluster.params['post']:+.3f}")
    print(f"  SE (cluster/state, 3 clusters): {m_cluster.bse['post']:.3f}   p={m_cluster.pvalues['post']:.3f}")
    print(f"  SE (HC1, non-clustered)       : {m_hc1.bse['post']:.3f}   p={m_hc1.pvalues['post']:.3f}")
    print(f"  implied handle change: {(np.exp(m_cluster.params['post']) - 1) * 100:.1f}%")
    print(f"  R-squared: {m_cluster.rsquared:.3f}   df_resid: {int(m_cluster.df_resid)}")

with open("../output/did_model_summary.txt", "w") as f:
    for label, (m_cluster, m_hc1) in results.items():
        f.write(f"{'='*100}\n{label}\n{'='*100}\n")
        f.write("--- cluster-robust (by state) ---\n")
        f.write(str(m_cluster.summary()))
        f.write("\n\n--- HC1 (non-clustered) ---\n")
        f.write(str(m_hc1.summary()))
        f.write("\n\n")

print("\nSaved: ../output/did_model_summary.txt")

STATIC PRE/POST SPECIFICATION (real data, N=224)

(1) Basic
  post coefficient : +0.323
  SE (cluster/state, 3 clusters): 0.073   p=0.000
  SE (HC1, non-clustered)       : 0.050   p=0.000
  implied handle change: 38.2%
  R-squared: 0.737   df_resid: 211

(2) + state-event linear trend
  post coefficient : +0.310
  SE (cluster/state, 3 clusters): 0.068   p=0.000
  SE (HC1, non-clustered)       : 0.071   p=0.000
  implied handle change: 36.3%
  R-squared: 0.778   df_resid: 199

Saved: ../output/did_model_summary.txt


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 12, but rank is 1
  warnings.warn('covariance of constraints does not have full '
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 1
  warnings.warn('covariance of constraints does not have full '
